In [0]:
def test_increasing_engagement(spark):
    from pyspark.sql.window import Window
    from pyspark.sql.functions import col, sum as _sum, lag, to_date, date_format

    data = [
        ("U1", "2024-01-01", 100),
        ("U1", "2024-02-01", 200),
        ("U1", "2024-03-01", 300)
    ]

    columns = ["user_id", "post_date", "likes"]
    df = spark.createDataFrame(data, columns) \
              .withColumn("post_date", to_date("post_date"))

    df = df.withColumn("month", date_format("post_date", "yyyy-MM"))

    monthly_df = df.groupBy("user_id", "month") \
                   .agg(_sum("likes").alias("monthly_likes"))

    window_spec = Window.partitionBy("user_id").orderBy("month")

    result_df = monthly_df.withColumn("prev1", lag("monthly_likes", 1).over(window_spec)) \
        .withColumn("prev2", lag("monthly_likes", 2).over(window_spec)) \
        .filter(
            (col("monthly_likes") > col("prev1")) &
            (col("prev1") > col("prev2"))
        )

    result = {r["user_id"] for r in result_df.collect()}

    expected = {"U1"}

    try:
        assert result == expected
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")
        print("Expected:", expected)
        print("Got:", result)

test_increasing_engagement(spark)